In [22]:
import os
import time
import torch
import torchaudio
from transformers import WhisperProcessor, WhisperForConditionalGeneration

In [23]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("No GPU detected. You are running on CPU.")

PyTorch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU count: 1


In [24]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [25]:
model_path = "."

# Load processor
processor = WhisperProcessor.from_pretrained(model_path)

# Load model (on GPU)
model = WhisperForConditionalGeneration.from_pretrained(model_path)
model.to(device)

print("Model and processor loaded successfully on GPU")

Model and processor loaded successfully on GPU


In [26]:
audio_folder = "audios"
audio_files = sorted(os.listdir(audio_folder))[:4]

waveforms = []

for file in audio_files:
    audio_path = os.path.join(audio_folder, file)
    
    waveform, sample_rate = torchaudio.load(audio_path)
    
    # Convert to mono
    waveform = waveform.mean(dim=0)
    
    # Resample to 16kHz
    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
    waveform = resampler(waveform)
    
    waveforms.append(waveform)
    
    print(f"Loaded: {file} | Shape: {waveform.shape}")

print(f"\nTotal loaded files: {len(waveforms)}")

Loaded: spontaneous-speech-en-1.mp3 | Shape: torch.Size([160704])
Loaded: spontaneous-speech-en-2.mp3 | Shape: torch.Size([190080])
Loaded: spontaneous-speech-en-3.mp3 | Shape: torch.Size([103104])
Loaded: spontaneous-speech-en-4.mp3 | Shape: torch.Size([82368])

Total loaded files: 4


In [27]:
inputs_list = []

for waveform in waveforms:
    inputs = processor(
        waveform.numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    )
    
    inputs_list.append(inputs)

print(f"Converted {len(inputs_list)} audio files to model inputs")

Converted 4 audio files to model inputs


In [28]:
predictions = []

for inputs in inputs_list:
    input_features = inputs["input_features"].to(device)
    
    with torch.no_grad():
        predicted_ids = model.generate(input_features)
    
    predictions.append(predicted_ids)

print(f"Generated predictions for {len(predictions)} files")

Generated predictions for 4 files


In [29]:
transcriptions = []

for predicted_ids in predictions:
    text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    transcriptions.append(text)

# Print results
for i, text in enumerate(transcriptions):
    print(f"\nFile {i+1}:")
    print(text)


File 1:
 I work out a lot, running, stretching, and taking care of myself through a healthy diet.

File 2:
 I'm really proud of my wife. She's quite a fantastic person, inspiring, and just full of really positive, good energy.

File 3:
 For whatever reason, I enjoy watching Nicholas Cage.

File 4:
 I told my teachers I wanted to be my own boss when I grew up.


In [30]:
for i, file in enumerate(audio_files):
    print(f"File {i+1}: {file}")

File 1: spontaneous-speech-en-1.mp3
File 2: spontaneous-speech-en-2.mp3
File 3: spontaneous-speech-en-3.mp3
File 4: spontaneous-speech-en-4.mp3


In [31]:
latencies = []

for inputs in inputs_list:
    input_features = inputs["input_features"].to(device)

    # Ensure accurate GPU timing
    torch.cuda.synchronize()
    start_time = time.time()

    with torch.no_grad():
        predicted_ids = model.generate(input_features)

    torch.cuda.synchronize()
    end_time = time.time()

    latency = end_time - start_time
    latencies.append(latency)

# Print results
for i, lat in enumerate(latencies):
    print(f"File {i+1} latency: {lat:.4f} seconds")

print(f"\nAverage latency: {sum(latencies)/len(latencies):.4f} seconds")

File 1 latency: 2.4905 seconds
File 2 latency: 2.2527 seconds
File 3 latency: 1.8498 seconds
File 4 latency: 2.0721 seconds

Average latency: 2.1663 seconds


In [34]:
import pandas as pd
from jiwer import wer, cer, Compose, ToLowerCase, RemovePunctuation, Strip

In [36]:
labels_path = "labels/ss-corpus-en.tsv"
df = pd.read_csv(labels_path, sep="\t")

# Define normalization pipeline
transform = Compose([
    ToLowerCase(),
    RemovePunctuation(),
    Strip()
])

ground_truths = []
valid_predictions = []

for file, pred in zip(audio_files, transcriptions):
    row = df[df["audio_file"] == file]
    
    if len(row) == 0:
        print(f"Skipping {file} (no transcript)")
        continue
    
    gt = row.iloc[0]["transcription"]
    
    ground_truths.append(gt)
    valid_predictions.append(pred)

# Compute metrics
wers = []
cers = []

cers = []
filtered_gt = []
filtered_pred = []

for gt, pred in zip(ground_truths, valid_predictions):
    # Normalize first
    gt_clean = transform(gt)
    pred_clean = transform(pred)
    
    # Skip empty cases
    if len(gt_clean.strip()) == 0:
        print("Skipping empty ground truth")
        continue
    
    w = wer(gt_clean, pred_clean)
    c = cer(gt_clean, pred_clean)
    
    wers.append(w)
    cers.append(c)
    filtered_gt.append(gt)
    filtered_pred.append(pred)

# Print results
for i, (gt, pred, w, c) in enumerate(zip(filtered_gt, filtered_pred, wers, cers)):
    print(f"\nFile {i+1}:")
    print("GT :", gt)
    print("Pred:", pred)
    print(f"WER: {w:.4f} | CER: {c:.4f}")


File 1:
GT : I workout a lot, running, stretching, and taking care of myself through <disfluency> a healthy diet
Pred:  I work out a lot, running, stretching, and taking care of myself through a healthy diet.
WER: 0.1875 | CER: 0.1458

File 2:
GT : I'm really proud of my wife, she's quite a fantastic person; <disfluency> inspiring, and just <disfluency> full of a really positive a good energy
Pred:  I'm really proud of my wife. She's quite a fantastic person, inspiring, and just full of really positive, good energy.
WER: 0.1667 | CER: 0.2128

File 3:
GT : for whatever reason, I enjoy watching Nicolas Cage
Pred:  For whatever reason, I enjoy watching Nicholas Cage.
WER: 0.1250 | CER: 0.0204

File 4:
GT : Told my teachers I wanted to be my own boss when I grew up
Pred:  I told my teachers I wanted to be my own boss when I grew up.
WER: 0.0714 | CER: 0.0345
